In [41]:
import pandas as pd
import numpy as np
import holidays
from sklearn.metrics.pairwise import haversine_distances
import h3

In [42]:
print("Loading needed datasets...")
taxi_agg = pd.read_parquet("../data/processed/aggregated_grid.parquet")
pois_cat_wide = pd.read_csv("../data/processed/chicago_pois_category_wide.csv")
print(f"Aggregated grid loaded. Shape: {taxi_agg.shape}")
print(f"POIs dataset loaded. Shape: {pois_cat_wide.shape}")

Loading needed datasets...
Aggregated grid loaded. Shape: (9272950, 12)
POIs dataset loaded. Shape: (806, 15)


In [43]:
# Find Null Values and add percentage of NaN values for each column
def generate_nan_report(df):
    is_na_df = df.isna().sum()
    is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
    is_na_df['Total Count'] = len(df)
    is_na_df['NaN Percentage'] = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100
    return is_na_df

# Check for any NaN values in all columns
print("NaN values before conversion:")
is_na_df = generate_nan_report(taxi_agg)
print(is_na_df)

NaN values before conversion:
                   NaN Count  Total Count  NaN Percentage
hour                       0      9272950        0.000000
h3_index                   0      9272950        0.000000
Total_Trip_Start           0      9272950        0.000000
Unique Taxis               0      9272950        0.000000
AvgTripSeconds             0      9272950        0.000000
AvgTripMiles               0      9272950        0.000000
AvgFare                    0      9272950        0.000000
MostCommonCompany          0      9272950        0.000000
CompanyCount               0      9272950        0.000000
PickupLatitude       8976030      9272950       96.797998
PickupLongitude      8976030      9272950       96.797998
Total_Trip_End             0      9272950        0.000000


In [44]:
# Fill Pickup Longitude and Latitude NaN values with h3 cell center coordinates
mask = taxi_agg['PickupLatitude'].isna() | taxi_agg['PickupLongitude'].isna()

# Compute centers only for the rows that need it (unique h3 indices -> cheap)
centers = taxi_agg.loc[mask, 'h3_index'].map(h3.cell_to_latlng)

taxi_agg.loc[mask, 'PickupLatitude']  = centers.str[0]
taxi_agg.loc[mask, 'PickupLongitude'] = centers.str[1]

# Check for any NaN values in all columns
print("NaN values before conversion:")
is_na_df = generate_nan_report(taxi_agg)
print(is_na_df)

NaN values before conversion:
                   NaN Count  Total Count  NaN Percentage
hour                       0      9272950             0.0
h3_index                   0      9272950             0.0
Total_Trip_Start           0      9272950             0.0
Unique Taxis               0      9272950             0.0
AvgTripSeconds             0      9272950             0.0
AvgTripMiles               0      9272950             0.0
AvgFare                    0      9272950             0.0
MostCommonCompany          0      9272950             0.0
CompanyCount               0      9272950             0.0
PickupLatitude             0      9272950             0.0
PickupLongitude            0      9272950             0.0
Total_Trip_End             0      9272950             0.0


## 1. Spatial Features

### TODO: Add more features for example: distance to nearest airport, public transit stops, etc.

In [45]:
# Calculate distance to Chicago Loop (downtown center)
# The Haversine distance is the shortest "as-the-crow-flies" 
# distance between two points on a sphere, calculated using their latitudes and longitudes
def new_haversine_distance_to_loop(lat1, lon1, ):
    # Use scikit-learn's haversine_distances function
    lat2=41.8781 # Chicago Loop latitude
    lon2=-87.6298 # Chicago Loop longitude
    # Convert latitudes and longitudes from degrees to radians
    coords_1 = np.radians(np.column_stack((lat1, lon1)))
    coords_2 = np.radians(np.array([[lat2, lon2]]))
    # Convert from radians to kilometers
    distances = haversine_distances(coords_1, coords_2) * 6371  
    return distances.flatten()

# Calculate distance to loop using Haversine function
taxi_agg['scikit_distance_to_loop'] = new_haversine_distance_to_loop(taxi_agg['PickupLatitude'], taxi_agg['PickupLongitude'])


## 2. Time Features

In [46]:
# Calendar features
taxi_agg['month'] = taxi_agg['hour'].dt.month
taxi_agg['day_of_week'] = taxi_agg['hour'].dt.weekday
taxi_agg['hour_of_day'] = taxi_agg['hour'].dt.hour
taxi_agg['is_weekend'] = (taxi_agg['day_of_week'] >= 5).astype(int)

# Cyclic temporal features
taxi_agg['hour_sin'] = np.sin(2 * np.pi * taxi_agg['hour_of_day'] / 24)
taxi_agg['hour_cos'] = np.cos(2 * np.pi * taxi_agg['hour_of_day'] / 24)
taxi_agg['month_sin'] = np.sin(2 * np.pi * taxi_agg['month'] / 12)
taxi_agg['month_cos'] = np.cos(2 * np.pi * taxi_agg['month'] / 12)

# Holidays feature
us_holidays = holidays.USA(years=taxi_agg['hour'].dt.year.unique(), state='IL')

taxi_agg['is_holiday'] = taxi_agg['hour'].dt.date.isin(us_holidays).astype(int)

## 3. Weather Features

### TODO: Add Weather features (e.g., temperature, precipitation) from the weather dataset, merged on hour and location.

## 4. POI Features

In [47]:
# Merge POI category features with the aggregated taxi dataset
taxi_agg = taxi_agg.merge(
    pois_cat_wide, 
    left_on='h3_index',
    right_on='h3_index',
    how='left'
)

# Fill NaN values in POI category columns with 0 and convert to int
taxi_agg['poi_cat_automotive'] = taxi_agg['poi_cat_automotive'].fillna(0).astype(int)
taxi_agg['poi_cat_entertainment'] = taxi_agg['poi_cat_entertainment'].fillna(0).astype(int)
taxi_agg['poi_cat_finance'] = taxi_agg['poi_cat_finance'].fillna(0).astype(int)
taxi_agg['poi_cat_food_drink'] = taxi_agg['poi_cat_food_drink'].fillna(0).astype(int)
taxi_agg['poi_cat_grocery'] = taxi_agg['poi_cat_grocery'].fillna(0).astype(int)
taxi_agg['poi_cat_health'] = taxi_agg['poi_cat_health'].fillna(0).astype(int)
taxi_agg['poi_cat_leisure_sports'] = taxi_agg['poi_cat_leisure_sports'].fillna(0).astype(int)
taxi_agg['poi_cat_lodging'] = taxi_agg['poi_cat_lodging'].fillna(0).astype(int)
taxi_agg['poi_cat_nightlife'] = taxi_agg['poi_cat_nightlife'].fillna(0).astype(int)
taxi_agg['poi_cat_services'] = taxi_agg['poi_cat_services'].fillna(0).astype(int)
taxi_agg['poi_cat_shopping'] = taxi_agg['poi_cat_shopping'].fillna(0).astype(int)
taxi_agg['poi_cat_transport'] = taxi_agg['poi_cat_transport'].fillna(0).astype(int)
taxi_agg['poi_cat_civic_community'] = taxi_agg['poi_cat_civic_community'].fillna(0).astype(int)
taxi_agg['poi_cat_education'] = taxi_agg['poi_cat_education'].fillna(0).astype(int)

# Check for any NaN values in all columns
print("NaN values before conversion:")
is_na_df = generate_nan_report(taxi_agg)
print(is_na_df)

NaN values before conversion:
                         NaN Count  Total Count  NaN Percentage
hour                             0      9272950             0.0
h3_index                         0      9272950             0.0
Total_Trip_Start                 0      9272950             0.0
Unique Taxis                     0      9272950             0.0
AvgTripSeconds                   0      9272950             0.0
AvgTripMiles                     0      9272950             0.0
AvgFare                          0      9272950             0.0
MostCommonCompany                0      9272950             0.0
CompanyCount                     0      9272950             0.0
PickupLatitude                   0      9272950             0.0
PickupLongitude                  0      9272950             0.0
Total_Trip_End                   0      9272950             0.0
scikit_distance_to_loop          0      9272950             0.0
month                            0      9272950             0.0
day_of_wee

## 5. Prediction Data Split for Train, Validation and Test
**Validation Strategy**: Use a robust **Temporal Split** (avoiding autokorrelation and data leakage from random splits) to evaluate out-of-sample predictive performance.

In [48]:
# Perform a temporal split (50/20/30) -> (Train/Validation/Test)
unique_dates = pd.Series(taxi_agg['hour'].dt.date.unique()).sort_values().reset_index(drop=True)

train_end = unique_dates.iloc[int(len(unique_dates) * 0.50)]
val_end   = unique_dates.iloc[int(len(unique_dates) * 0.70)]

print(f"Train:      until {train_end}")
print(f"Validation: until {val_end}")

df_train = taxi_agg[taxi_agg['hour'].dt.date < train_end]
df_val   = taxi_agg[(taxi_agg['hour'].dt.date >= train_end) & (taxi_agg['hour'].dt.date < val_end)]
df_test  = taxi_agg[taxi_agg['hour'].dt.date >= val_end]

print(f"Train: {len(df_train)} rows | Val: {len(df_val)} rows | Test: {len(df_test)} rows")

Train:      until 2025-03-02
Validation: until 2025-08-19
Train: 4641696 rows | Val: 1852320 rows | Test: 2778934 rows


In [49]:
# Historical base demand — computed ONLY from train (no leakage)
keys = ["h3_index", "hour_of_day", "is_weekend"]
base = (df_train.groupby(keys)["Total_Trip_Start"]
        .mean().rename("base_demand").reset_index())
global_base = df_train["Total_Trip_Start"].mean()

def add_base_demand(df):
    df = df.merge(base, on=keys, how="left")
    # unseen keys -> global mean
    df["base_demand"] = df["base_demand"].fillna(global_base)  
    return df

df_train = add_base_demand(df_train)
df_val   = add_base_demand(df_val)
df_test  = add_base_demand(df_test)

In [50]:
# Export aggregated feature dataset for modeling
taxi_agg.to_parquet("../data/processed/prediction_features.parquet")

# Export the splits to parquets:
df_train.to_parquet("../data/prediction_split/df_train.parquet", index=False)
df_val.to_parquet("../data/prediction_split/df_val.parquet",     index=False)
df_test.to_parquet("../data/prediction_split/df_test.parquet",   index=False)